In [1]:
from pathlib import Path
from typing import Union
import pandas as pd
import numpy as np
import time
import itertools

from infer_subc.core.file_io import (read_czi_image,
                                     read_tiff_image,
                                     list_image_files)
from infer_subc.utils.batch import (find_segmentation_tiff_files)
from infer_subc.utils.stats import (get_org_morphology_3D, 
                                    get_region_morphology_3D,
                                    surface_area_from_props)
from infer_subc.core.img import *
from skimage.measure import regionprops_table

C:\Users\zscoman\AppData\Local\Temp\ipykernel_16756\2513996729.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
def make_dict(list_obj_names: list[str],
               list_obj_segs: list[np.ndarray]):
    organelle_segs = {}                                                     
    for idx, name in enumerate(list_obj_names):                                  
        if name == 'ER':                                                    
            organelle_segs[name]=(list_obj_segs[idx]>0).astype(np.uint16)        
        else:                                                       
            organelle_segs[name]=list_obj_segs[idx]
    return organelle_segs


In [3]:

def all_combo(list_obj_names: list[str], splitter: str="X"):
    all_pos = []
    for n in list(map(lambda x:x+2, (range(len(list_obj_names)-1)))):
        all_pos += itertools.combinations(list_obj_names, n)
    possib = [splitter.join(inter) for inter in all_pos]
    return possib


In [4]:
def create_overlap(orgs:str,
                   organelle_segs: dict[str:np.ndarray],
                   splitter: str="X") -> tuple[np.ndarray, np.ndarray]: 
    ##########################################
    ## CREATE OVERLAP
    ##########################################
    site = np.ones_like(organelle_segs[orgs.split(splitter)[0]])
    for org in orgs.split(splitter):
        b = organelle_segs[org]             
        valid = (b>0)*(site>0)
        digit = len(str(np.max(site)))      
        site = (b*(10**(digit)))+site       
        site[valid.astype(bool)==False]=0   
        site = label(site)             
    return site

In [5]:
def find_non_redundant_overlaps(site: np.ndarray,
                                orgs: str,
                                organelle_segs: dict[str:np.ndarray],
                                splitter: str="X"):
    ##########################################
    ## DETERMINE REDUNDANT OVERLAPS
    ##########################################
    LOc_NR = site.copy()                      
    for org, val in organelle_segs.items():         
        if (org not in orgs.split(splitter)
            and np.any(site.astype(int)*val.astype(int))):
            print(f"Examining {orgs} Higher Order Interactions With {org}...", end="\r")               
            digit = len(str(np.max(val)))           
            valid = (LOc_NR>0)*(val>0)              
            HOc = (LOc_NR*(10**(digit)))+val        
            HOc[valid.astype(bool)==False]=0        
            HOc = label(HOc) 
            maxi = len(np.unique(LOc_NR[HOc>0]))                       
            for num, id in enumerate(np.unique(site[HOc > 0])):
                per = round((100*((num+1)/maxi)), 2)   
                LOc_NR[LOc_NR==id] = 0
                print(f"Examining {orgs} Higher Order Interactions With {org}: {per}% complete", end="\r")
            print(f"Examining {orgs} Higher Order Interactions With {org}: {per}% complete")     
    return LOc_NR


In [6]:
def interaction_metric_analysis(overlap_ID: str,
                                list_obj_names: list[str],
                                list_obj_segs: list[np.ndarray],
                                mask: np.ndarray,
                                splitter: str="X",
                                scale: Union[tuple, None]=None,
                                return_site: bool=False):
    """
    collect volumentric measurements of intersection between n organelle types

    Parameters
    ------------
    overlap_ID: str
        a value used to describe the organelles present in the overlap that can be divided by the splitter value
    org_dict: dict
        a dictionary of all object segmentations assigned to keys with their objects
    mask: np.ndarray
        3D (ZYX) binary mask of the area to measure interactions from
    splitter: str
        a value used to separate the overlap_ID to determine objects present in overlap
    scale: tuple
        a value present in the metadata determining the scale of the (ZYX) axis
    include_dist:bool=False
        *optional*
        True = include the XY and Z distribution measurements of the overlap sites within the masked region 
        (utilizing the functions get_XY_distribution() and get_Z_distribution() from Infer-subc)
        False = do not include distirbution measurements
    dist_centering_obj: Union[np.ndarray, None]=None
        ONLY NEEDED IF include_dist=True; if None, the center of the mask will be used
        3D (ZYX) np.ndarray containing the object to use for centering the XY distribution mask
    dist_num_bins: Union[int, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is 5
    dist_zernike_degrees: Unions[int, None]=None,
        ONLY NEEDED IF include_dist=True; if None, the zernike share measurements will not be included in the distribution
        the number of zernike degrees to include for the zernike shape descriptors
    dist_center_on: Union[bool, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is False
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    dist_keep_center_as_bin: Union[bool, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is True
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins


    Regionprops measurements:
    ------------------------
    ['label',
    'centroid',
    'bbox',
    'area',
    'equivalent_diameter',
    'extent',
    'feret_diameter_max',
    'euler_number',
    'convex_area',
    'solidity',
    'axis_major_length',
    'axis_minor_length']

    Additional measurements:
    ----------------------
    ['surface_area']

    
    Returns
    -------------
    pandas dataframe of containing regionprops measurements (columns) for each overlap region (rows)
    
    """
    #########################
    ## CREATE ORG_DICT
    #########################
    org_dict = make_dict(list_obj_names, list_obj_segs)


    #########################
    ## CREATE OVERLAP REGIONS
    #########################
    # run create overlap function
    site = create_overlap(overlap_ID, org_dict, splitter)

    #############################################################################################
    #assert the nth order overlap to within the cellmask
    labels = label(apply_mask(site, mask)).astype(int)


    ##########################################
    ## CREATE LIST OF REGIONPROPS MEASUREMENTS
    ##########################################
    # start with LABEL
    properties = ["label"]

    # add position
    properties += ["centroid", "bbox"]

    # add area
    properties += ["area", "equivalent_diameter"] # "num_pixels", 

    # add shape measurements - NOTE: can't include minor axis measure because some of the contact sites are only one pixel
    properties += ["extent", "euler_number", "solidity", "axis_major_length", "slice"] # "feret_diameter_max",  , "axis_minor_length"
    

    ##################
    ## RUN REGIONPROPS
    ##################
    props = regionprops_table(labels, 
                              intensity_image=None, 
                              properties=properties, 
                              extra_properties=None, 
                              spacing=scale)

    ##################################################################
    ## RUN SURFACE AREA FUNCTION SEPARATELY AND APPEND THE PROPS_TABLE
    ##################################################################
    surface_area_tab = pd.DataFrame(surface_area_from_props(labels, props, scale))

    #################################################################################################


    ########################################################
    ## LIST WHICH ORGANELLES ARE INVOLVED IN THE INTERACTION
    ########################################################
    over_inv = []
    involved = overlap_ID.split(splitter)
    indexes = dict.fromkeys(involved, [])
    indexes[overlap_ID] = []

    for index, l in enumerate(props["label"]):
        over_inv.clear()
        for org in involved:
            volume = labels[props["slice"][index]]
            lorg = org_dict[org][props["slice"][index]]
            volume = volume==l
            lorg = lorg[volume]                                 
            all_inv = np.unique(lorg[lorg>0]).tolist()          
            if len(all_inv) != 1:
                print(f"we have an error.  as-> {all_inv}")
            indexes[org].append(all_inv[0])
            over_inv.append(f"{all_inv[0]}")
        indexes[overlap_ID].append('_'.join(over_inv))

        
    ##################################################
    ## CREATE COMBINED DATAFRAME OF THE QUANTIFICATION
    ##################################################
    props_table = pd.DataFrame(props)
    props_table.rename(columns={'label': 'idx'}, inplace=True)
    props_table.drop(columns=['slice'], inplace=True)
    props_table.insert(0, 'label',value=indexes[overlap_ID])
    props_table.insert(0, "object", overlap_ID)
    props_table.rename(columns={"area": "volume"}, inplace=True)
    props_table.insert(11, "surface_area", surface_area_tab)
    props_table.insert(13, "SA_to_volume_ratio", 
    props_table["surface_area"].div(props_table["volume"]))
    if scale is not None:
        round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
        props_table.insert(loc=2, column="scale", value=f"{round_scale}")
    else: 
        props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(labels.ndim))}")


    ######################################################
    ## optional: DISTRIBUTION OF INTERACTION MEASUREMENTS
    ######################################################
    indexes.clear()
    if return_site:
        return site, props_table 
    else:
        return props_table

In [22]:
def get_interaction_metrics_3D(list_obj_names: list[str],
                               list_obj_segs: list[np.ndarray],
                               mask: np.ndarray,
                               splitter: str="X",
                               scale: Union[tuple, None]=None):
    """
    collect volumentric measurements of intersection between n, n+1, n+2... organelle types for an entire cell

    Parameters
    ------------
    list_obj_names: list
        a list of the names of objects used in making overlaps
    list_obj_segs: list
        a list of the segmentations of the objects used in making overlaps
    mask: np.ndarray
        3D (ZYX) binary mask of the area to measure overlaps from
    splitter: str="X"
        a value used to separate the organelles in their IDs
    scale: tuple
        3D (ZYX) assignment for the scaling of each axis
        include_dist:bool=False
        *optional*
        True = include the XY and Z distribution measurements of the overlaps sites within the masked region 
        (utilizing the functions get_XY_distribution() and get_Z_distribution() from Infer-subc)
        False = do not include distirbution measurements
    dist_centering_obj: Union[np.ndarray, None]=None
        ONLY NEEDED IF include_dist=True; if None, the center of the mask will be used
        3D (ZYX) np.ndarray containing the object to use for centering the XY distribution mask
    dist_num_bins: Union[int, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is 5
    dist_zernike_degrees: Unions[int, None]=None,
        ONLY NEEDED IF include_dist=True; if None, the zernike share measurements will not be included in the distribution
        the number of zernike degrees to include for the zernike shape descriptors
    dist_center_on: Union[bool, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is False
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    dist_keep_center_as_bin: Union[bool, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is True
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins

    Regionprops measurements:
    ------------------------
    ['label',
    'centroid',
    'bbox',
    'area',
    'equivalent_diameter',
    'extent',
    'feret_diameter_max',
    'euler_number',
    'convex_area',
    'solidity',
    'axis_major_length',
    'axis_minor_length']

    Additional measurements:
    ----------------------
    ['surface_area']

    
    Returns
    -------------
    pandas dataframe of containing regionprops measurements (columns) for each overlap region (rows)
    """

    ########################
    ## CREATE ORGANELLE DICT
    ########################
    organelle_segs = make_dict(list_obj_names, list_obj_segs)                                                 
    
    #########################
    ## LIST POSSIBLE OVERLAPS
    #########################
    possib = all_combo(list_obj_names, splitter)

    #######################
    ## ANALYZE ALL OVERLAPS
    #######################
    inter_tabs=[]
    for inter in possib:
        site, inter_tab = interaction_metric_analysis(overlap_ID=inter,
                                                      list_obj_names = list_obj_names,
                                                      list_obj_segs = list_obj_segs,
                                                      mask=mask,
                                                      splitter=splitter,
                                                      scale=scale,
                                                      return_site=True)
        LOi_NR = find_non_redundant_overlaps(site, inter, organelle_segs, splitter)
        LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site
        redundancy = inter_tab['idx'].isin(np.unique(LOi_NR[LOi_NR>0]).tolist())
        inter_tab.insert(2, "in_higher_order", list(map(bool, ~redundancy)))
        inter_tab.drop(columns=['idx'], inplace=True)
        inter_tabs.append(inter_tab)
    return inter_tabs


In [23]:
def make_all_metrics_tables(source_file: str,
                             list_obj_names: List[str],
                             list_obj_segs: List[np.ndarray],
                             list_intensity_img: List[np.ndarray],
                             list_region_names: List[str],
                             list_region_segs: List[np.ndarray],
                             mask: str,
                             scale: Union[tuple,None] = None):
    """
    Measure the composition, morphology, distribution, and interactions of multiple organelles in a cell

    Parameters:
    ----------
    source_file: str
        file path; this is used for recorder keeping of the file name in the output data tables
    list_obj_names: List[str]
        a list of object names (strings) that will be measured; this should match the order in list_obj_segs
    list_obj_segs: List[np.ndarray]
        a list of 3D (ZYX) segmentation np.ndarrays that will be measured per cell; the order should match the list_obj_names 
    list_intensity_img: List[np.ndarray]
        a list of 3D (ZYX) grayscale np.ndarrays that will be used to measure fluoresence intensity in each region and object
    list_region_names: List[str]
        a list of region names (strings); these should include the mask (entire region being measured - usually the cell) 
        and other sub-mask regions from which we can meausure the objects in (ex - nucleus, neurites, soma, etc.). It should 
        also include the centering object used when created the XY distribution bins.
        The order should match the list_region_segs
    list_region_segs: List[np.ndarray]
        a list of 3D (ZYX) binary np.ndarrays of the region masks; the order should match the list_region_names.
    mask: str
        a str of which region name (contained in the list_region_names list) should be used as the main mask (e.g., cell mask)
    dist_centering_obj:str
        a str of which region name (contained in the list_region_names list) should be used as the centering object in 
        get_XY_distribution()
    dist_num_bins: int
        the number of concentric rings to draw between the centering object and edge of the mask in get_XY_distribution()
    dist_center_on: bool=False,
        for get_XY_distribution:
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    dist_keep_center_as_bin: bool=True
        for get_XY_distribution:
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins
    dist_zernike_degrees: Union[int, None]=None
        for get_XY_distribution:
        the number of zernike degrees to include for the zernike shape descriptors; if None, the zernike measurements will not 
        be included in the output
    scale: Union[tuple,None] = None
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
    include_interaction_dist:bool=True
        whether to include the distribution of overlap sites in get_interaction_metrics_3d(); True = include interaction distribution

    Returns:
    ----------
    4 Dataframes of measurements of organelle morphology, region morphology, overlap morphology, and organelle/interaction distributions

    """
    start = time.time()
    count = 0

    # segmentation image for all masking steps below
    mask = list_region_segs[list_region_names.index(mask)]

    ######################
    # measure cell regions
    ######################
    # create np.ndarray of intensity images
    raw_image = np.stack(list_intensity_img)
    
    # container for region data
    region_tabs = []
    for r, r_name in enumerate(list_region_names):
        region = list_region_segs[r]
        region_metrics = get_region_morphology_3D(region_seg=region, 
                                                  region_name=r_name,
                                                  channel_names=list_obj_names,
                                                  intensity_img=raw_image, 
                                                  mask=mask,
                                                  scale=scale)
        region_tabs.append(region_metrics)

    ##############################################################
    # loop through all organelles to collect measurements for each
    ##############################################################
    # containers to collect per organelle information
    org_tabs = []
    dist_tabs = []
    XY_bins = []
    XY_wedges = []

    for j, target in enumerate(list_obj_names):
        # organelle intensity image
        org_img = list_intensity_img[j]

        # organelle segmentation
        if target == 'ER':
            # ensure ER is only one object
            org_obj = (list_obj_segs[j] > 0).astype(np.uint16)
        else:
            org_obj = list_obj_segs[j]

        ##########################################################
        # measure organelle morphology & number of objs overlapping
        ##########################################################
        org_metrics = get_org_morphology_3D(segmentation_img=org_obj, 
                                            seg_name=target,
                                            intensity_img=org_img, 
                                            mask=mask,
                                            scale=scale)

        ### org_metrics.insert(loc=0,column='cell',value=1) 
        # ^^^ saving this thought for later when someone might have more than one cell per image.
        # Not sure how they analysis process would fit in our pipelines as they exist now. 
        # Maybe here, iterating though the index of the masks above all of this and using that index as the cell number?

        org_tabs.append(org_metrics)


    ###########################################
    # collect non-redundant interaction metrics 
    ###########################################
    if (len(list_obj_names)>2):
        interaction_tabs = get_interaction_metrics_3D(list_obj_names=list_obj_names,
                                                  list_obj_segs=list_obj_segs,
                                                  mask=mask,
                                                  scale=scale)


    ###########################################
    # combine all tabs into one table per type:
    ###########################################
    final_org_tab = pd.concat(org_tabs, ignore_index=True)
    final_org_tab.insert(loc=0,column='image_name',value=source_file.stem)

    final_interaction_tab = pd.concat(interaction_tabs, ignore_index=True)
    final_interaction_tab.insert(loc=0,column='image_name',value=source_file.stem)

    final_region_tab = pd.concat(region_tabs, ignore_index=True)
    final_region_tab.insert(loc=0,column='image_name',value=source_file.stem)

    end = time.time()
    print(f"It took {(end-start)/60} minutes to quantify one image.")
    return final_org_tab, final_interaction_tab, final_region_tab

In [24]:
def batch_process_quantification(out_file_name: str,
                                  seg_path: Union[Path,str],
                                  out_path: Union[Path, str], 
                                  raw_path: Union[Path,str], 
                                  raw_file_type: str,
                                  organelle_names: List[str],
                                  organelle_channels: List[int],
                                  region_names: List[str],
                                  masks_file_name: str,
                                  mask: str,
                                  scale:bool=True,
                                  seg_suffix:Union[str, None]=None) -> int :
    """  
    batch process segmentation quantification (morphology, distribution, contacts); this function is currently optimized to process images from one file folder per image type (e.g., raw, segmentation)
    the output csv files are saved to the indicated out_path folder

    Parameters:
    ----------
    out_file_name: str
        the prefix to use when naming the output datatables
    seg_path: Union[Path,str]
        Path or str to the folder that contains the segmentation tiff files
    out_path: Union[Path, str]
        Path or str to the folder that the output datatables will be saved to
    raw_path: Union[Path,str]
        Path or str to the folder that contains the raw image files
    raw_file_type: str
        the file type of the raw data; ex - ".tiff", ".czi"
    organelle_names: List[str]
        a list of all organelle names that will be analyzed; the names should be the same as the suffix used to name each of the tiff segmentation files
        Note: the intensity measurements collect per region (from get_region_morphology_3D function) will only be from channels associated to these organelles 
    organelle_channels: List[int]
        a list of channel indices associated to respective organelle staining in the raw image; the indices should listed in same order in which the respective segmentation name is listed in organelle_names
    region_names: List[str]
        a list of regions, or masks, to measure; the order should correlate to the order of the channels in the "masks" output segmentation file
    masks_file_name: str
        the suffix of the "masks" segmentation file; ex- "masks_B", "masks", etc.
        this function currently does not accept indivial region segmentations 
    mask: str
        the name of the region to use as the mask when measuring the organelles; this should be one of the names listed in regions list; usually this will be the "cell" mask
    dist_centering_obj:str
        the name of the region or object to use as the centering object in the get_XY_distribution function
    dist_num_bins: int
        the number of bins for the get_XY_distribution function
    dist_center_on: bool=False,
        for get_XY_distribution:
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    dist_keep_center_as_bin: bool=True
        for get_XY_distribution:
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins
    dist_zernike_degrees: Union[int, None]=None
        for get_XY_distribution:
        the number of zernike degrees to include for the zernike shape descriptors; if None, the zernike measurements will not 
        be included in the output
    include_contact_dist:bool=True
        whether to include the distribution of contact sites in get_contact_metrics_3d(); True = include contact distribution
    scale:bool=True
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
    seg_suffix:Union[str, None]=None
        any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix
    


    Returns:
    ----------
    count: int
        the number of images processed
        
    """
    start = time.time()
    count = 0

    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(out_path, str): out_path = Path(out_path)
    
    if not Path.exists(out_path):
        Path.mkdir(out_path)
        print(f"making {out_path}")
    
    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)

    # list of segmentation files to collect
    segs_to_collect = organelle_names + masks_file_name

    # containers to collect data tabels
    org_tabs = []
    contact_tabs = []
    region_tabs = []
    for img_f in img_file_list:
        count = count + 1
        filez = find_segmentation_tiff_files(img_f, segs_to_collect, seg_path, seg_suffix)

        # read in raw file and metadata
        img_data, meta_dict = read_czi_image(filez["raw"])

        # create intensities from raw file as list based on the channel order provided
        intensities = [img_data[ch] for ch in organelle_channels]

        # define the scale
        if scale is True:
            scale_tup = meta_dict['scale']
        else:
            scale_tup = None

        # load regions as a list based on order in list (should match order in "masks" file)
        masks = [read_tiff_image(filez[mask]) for mask in masks_file_name]
        regions = [masks[r] for r, region in enumerate(region_names)]

        # store organelle images as list
        organelles = [read_tiff_image(filez[org]) for org in organelle_names]

        org_metrics, contact_metrics, region_metrics = make_all_metrics_tables(source_file=img_f,
                                                                               list_obj_names=organelle_names,
                                                                               list_obj_segs=organelles,
                                                                               list_intensity_img=intensities, 
                                                                               list_region_names=region_names,
                                                                               list_region_segs=regions, 
                                                                               mask=mask,
                                                                               scale=scale_tup)

        org_tabs.append(org_metrics)
        contact_tabs.append(contact_metrics)
        region_tabs.append(region_metrics)
        end2 = time.time()
        print(f"Completed processing for {count} images in {(end2-start)/60} mins.")

    final_org = pd.concat(org_tabs, ignore_index=True)
    final_contact = pd.concat(contact_tabs, ignore_index=True)
    final_region = pd.concat(region_tabs, ignore_index=True)

    org_csv_path = out_path / f"{out_file_name}_organelles.csv"
    final_org.to_csv(org_csv_path)

    contact_csv_path = out_path / f"{out_file_name}_contacts.csv"
    final_contact.to_csv(contact_csv_path)

    region_csv_path = out_path / f"{out_file_name}_regions.csv"
    final_region.to_csv(region_csv_path)

    end = time.time()
    print(f"Quantification for {count} files is COMPLETE! Files saved to '{out_path}'.")
    print(f"It took {(end - start)/60} minutes to quantify these files.")
    return count

In [25]:
seg=batch_process_quantification(out_file_name= "neurite_checks_neurites",
                                 seg_path="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/segmentations",
                                 out_path="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/outputs", 
                                 raw_path="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/raw",
                                 raw_file_type = ".tiff",
                                 organelle_names = ['LD', 'ER', 'golgi', 'lyso', 'mito', 'perox'],
                                 organelle_channels= [0, 6, 4, 2, 3, 5],
                                 region_names= ['nuc', 'neurites'],
                                 masks_file_name= ['nuc', 'neurites'],
                                 mask= 'neurites',
                                 scale=True,
                                 seg_suffix="-")

c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:430: UserWarning: Failed to get convex hull image. Returning empty image, see error message below:
QH6214 qhull input error: not enough points(1) to construct initial simplex (need 4)

While executing:  | qhull i Qt
Options selected for Qhull 2019.1.r 2019/06/21:
  run-id 1262037050  incidence  Qtriangulate  _pre-merge  _zero-centrum
  _maxoutside  0

  return convex_hull_image(self.image)
c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:629: RuntimeWarning: divide by zero encountered in scalar divide
  return self.area / self.area_convex
c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:430: UserWarning: Failed to get convex hull image. Returning empty image, see error message below:
QH6013 qhull input error: input is less than 3-dimensional since all points have the same x coordinate    0

While executing: 

Examining ERXgolgi Higher Order Interactions With lyso: 100.0% complete
Examining ERXgolgi Higher Order Interactions With mito: 100.0% complete
Examining ERXgolgi Higher Order Interactions With perox: 100.0% complete


c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:430: UserWarning: Failed to get convex hull image. Returning empty image, see error message below:
QH6013 qhull input error: input is less than 3-dimensional since all points have the same x coordinate    0

While executing:  | qhull i Qt
Options selected for Qhull 2019.1.r 2019/06/21:
  run-id 1262541260  incidence  Qtriangulate  _pre-merge  _zero-centrum
  _max-width  4  Error-roundoff 5.5e-15  _one-merge 3.9e-14
  _near-inside 1.9e-13  Visible-distance 1.1e-14  U-max-coplanar 1.1e-14
  Width-outside 2.2e-14  _wide-facet 6.7e-14  _maxoutside 4.4e-14

  return convex_hull_image(self.image)
c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:629: RuntimeWarning: divide by zero encountered in scalar divide
  return self.area / self.area_convex
c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:430: UserWarning: Fa

Examining ERXlyso Higher Order Interactions With golgi: 100.0% complete
Examining ERXlyso Higher Order Interactions With mito: 100.0% complete
Examining ERXlyso Higher Order Interactions With perox: 100.0% complete


c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:430: UserWarning: Failed to get convex hull image. Returning empty image, see error message below:
QH6214 qhull input error: not enough points(2) to construct initial simplex (need 4)

While executing:  | qhull i Qt
Options selected for Qhull 2019.1.r 2019/06/21:
  run-id 1262625295  incidence  Qtriangulate  _pre-merge  _zero-centrum
  _maxoutside  0

  return convex_hull_image(self.image)
c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:629: RuntimeWarning: divide by zero encountered in scalar divide
  return self.area / self.area_convex
c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:430: UserWarning: Failed to get convex hull image. Returning empty image, see error message below:
QH6013 qhull input error: input is less than 3-dimensional since all points have the same x coordinate    0

While executing: 

...

Examining golgiXmitoXperox Higher Order Interactions With ER: 100.0% complete


c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:430: UserWarning: Failed to get convex hull image. Returning empty image, see error message below:
QH6214 qhull input error: not enough points(1) to construct initial simplex (need 4)

While executing:  | qhull i Qt
Options selected for Qhull 2019.1.r 2019/06/21:
  run-id 1264003469  incidence  Qtriangulate  _pre-merge  _zero-centrum
  _maxoutside  0

  return convex_hull_image(self.image)
c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:629: RuntimeWarning: divide by zero encountered in scalar divide
  return self.area / self.area_convex


Examining lysoXmitoXperox Higher Order Interactions With ER: 100.0% complete


c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:430: UserWarning: Failed to get convex hull image. Returning empty image, see error message below:
QH6214 qhull input error: not enough points(1) to construct initial simplex (need 4)

While executing:  | qhull i Qt
Options selected for Qhull 2019.1.r 2019/06/21:
  run-id 1264053890  incidence  Qtriangulate  _pre-merge  _zero-centrum
  _maxoutside  0

  return convex_hull_image(self.image)
c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:629: RuntimeWarning: divide by zero encountered in scalar divide
  return self.area / self.area_convex
c:\Users\zscoman\Anaconda3\envs\infer-subc\lib\site-packages\skimage\measure\_regionprops.py:430: UserWarning: Failed to get convex hull image. Returning empty image, see error message below:
QH6214 qhull input error: not enough points(1) to construct initial simplex (need 4)

While executing:  | qhull i Qt
Options s

It took 2.8251261552174887 minutes to quantify one image.
Completed processing for 1 images in 2.855206795533498 mins.
Quantification for 1 files is COMPLETE! Files saved to 'C:\Users\zscoman\Documents\Python Scripts\Infer-subc-2D\neurites\outputs'.
It took 2.8552158872286477 minutes to quantify these files.
